# CubiCasa5K + CubiGraph5K parser test

Upload a floor-plan **PNG/JPG** to obtain CubiCasa5K semantic segmentation and vector polygons. Then upload a CubiCasa-compatible, semantically annotated **`model.svg`** to generate the CubiGraph5K room-relation graph.

> **Important:** CubiGraph5K consumes annotated SVG geometry, not a raster image. It cannot infer rooms or doors from a PNG/JPG by itself. Use a CubiCasa5K source `model.svg` (for example, from the CubiCasa5K dataset) for the CubiGraph section.

In Colab, select **Runtime → Change runtime type → T4 GPU** before running the notebook.

## 1. Setup

This downloads the official source repositories and the published CubiCasa5K checkpoint (~203 MB). Run once per fresh Colab runtime.

In [ ]:
!rm -rf /content/CubiCasa5k /content/CubiGraph5K
!git clone --depth 1 https://github.com/CubiCasa/CubiCasa5k.git /content/CubiCasa5k
!git clone --depth 1 https://github.com/luyueheng/CubiGraph5K.git /content/CubiGraph5K
!pip -q install gdown beautifulsoup4 lxml shapely networkx svgwrite
!gdown --fuzzy 'https://drive.google.com/uc?id=1gRB7ez1e4H7a9Y09lLqRuna0luZO5VRK' -O /content/CubiCasa5k/model_best_val_loss_var.pkl

import os, sys, json, shutil
from pathlib import Path
WORKDIR = Path('/content/floorplan_parser_output')
WORKDIR.mkdir(exist_ok=True)
print('Working directory:', WORKDIR)

## 2. CubiCasa5K — upload a floor-plan image

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image

uploaded = files.upload()  # choose one PNG, JPG, or JPEG floor-plan image
assert uploaded, 'Please upload a floor-plan image.'
image_name, image_bytes = next(iter(uploaded.items()))
image_path = WORKDIR / image_name
image_path.write_bytes(image_bytes)
source_image = Image.open(image_path).convert('RGB')
display(source_image)
print('Saved:', image_path, '| original size:', source_image.size)

## 3. Load the published CubiCasa5K model

The original project targets an older PyTorch release. This cell keeps its model architecture and checkpoint, but uses the current Colab device API.

In [ ]:
%cd /content/CubiCasa5k
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from floortrans.models import get_model
from floortrans.loaders import RotateNTurns
from floortrans.plotting import discrete_cmap, polygons_to_image
from floortrans.post_prosessing import split_prediction, get_polygons

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Enable a GPU runtime in Colab before continuing.'
split = [21, 12, 11]
room_classes = ['Background', 'Outdoor', 'Wall', 'Kitchen', 'Living Room', 'Bed Room', 'Bath', 'Entry', 'Railing', 'Storage', 'Garage', 'Undefined']
icon_classes = ['No Icon', 'Window', 'Door', 'Closet', 'Electrical Appliance', 'Toilet', 'Sink', 'Sauna Bench', 'Fire Place', 'Bathtub', 'Chimney']

model = get_model('hg_furukawa_original', 51)
model.conv4_ = torch.nn.Conv2d(256, 44, bias=True, kernel_size=1)
model.upsample = torch.nn.ConvTranspose2d(44, 44, kernel_size=4, stride=4)
checkpoint = torch.load('model_best_val_loss_var.pkl', map_location=device)
model.load_state_dict(checkpoint['model_state'])
model.to(device).eval()
rot = RotateNTurns()
discrete_cmap()
print('Model loaded on', device)

## 4. Run CubiCasa5K parsing

For reliability and GPU memory use, the longest side is limited to 1,024 pixels, then padded to a multiple of 32. The result is mapped back to that processed image size.

In [ ]:
MAX_SIDE = 1024
img = source_image.copy()
scale = min(1.0, MAX_SIDE / max(img.size))
resized_size = (round(img.width * scale), round(img.height * scale))
img = img.resize(resized_size, Image.Resampling.LANCZOS)
padded_w = (img.width + 31) // 32 * 32
padded_h = (img.height + 31) // 32 * 32
canvas = Image.new('RGB', (padded_w, padded_h), 'white')
canvas.paste(img, (0, 0))
x = (np.asarray(canvas).astype(np.float32) / 255.0 - 0.5) * 2.0
image_tensor = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).to(device)
``
with torch.no_grad():
    predictions = []
    for forward, backward in [(0, 0), (1, -1), (2, 2), (-1, 1)]:
        pred = model(rot(image_tensor, 'tensor', forward))
        pred = rot(pred, 'tensor', backward)
        pred = rot(pred, 'points', backward)
        pred = F.interpolate(pred, size=(padded_h, padded_w), mode='bilinear', align_corners=True)
        predictions.append(pred)
    prediction = torch.stack(predictions).mean(0).cpu()

heatmaps, rooms, icons = split_prediction(prediction, (padded_h, padded_w), split)
polygons, types, room_polygons, room_types = get_polygons((heatmaps, rooms, icons), 0.2, [1, 2])
room_seg, icon_seg = polygons_to_image(polygons, types, room_polygons, room_types, padded_h, padded_w)

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
axes[0].imshow(canvas); axes[0].set_title('Processed input'); axes[0].axis('off')
axes[1].imshow(room_seg, cmap='rooms', vmin=0, vmax=10.9); axes[1].set_title('CubiCasa5K rooms / walls'); axes[1].axis('off')
axes[2].imshow(icon_seg, cmap='icons', vmin=0, vmax=10.9); axes[2].set_title('CubiCasa5K icons'); axes[2].axis('off')
plt.tight_layout()

np.savez_compressed(WORKDIR / 'cubicasa5k_prediction.npz', room_seg=room_seg, icon_seg=icon_seg)
print('Saved segmentation masks:', WORKDIR / 'cubicasa5k_prediction.npz')

## 5. CubiGraph5K — upload an annotated `model.svg`

Upload the `model.svg` for a CubiCasa5K dataset example. The SVG must contain `g class="Space"` room groups and `g class="Threshold"` door groups. CubiGraph labels edges as `1 = adjacent` and `2 = connected through a door`.

In [ ]:
uploaded_svg = files.upload()  # choose model.svg, not a PNG/JPG
assert uploaded_svg, 'Please upload a CubiCasa-style model.svg file.'
svg_name, svg_bytes = next(iter(uploaded_svg.items()))
assert svg_name.lower().endswith('.svg'), 'CubiGraph5K requires an SVG file.'
svg_path = WORKDIR / 'model.svg'
svg_path.write_bytes(svg_bytes)
print('Saved:', svg_path)

In [ ]:
sys.path.insert(0, '/content/CubiGraph5K/src')
from bs4 import BeautifulSoup
from plan import Plan
from IPython.display import SVG, display

soup = BeautifulSoup(svg_path.read_text(encoding='utf-8'), 'lxml')
plan = Plan(soup.find('svg'))
plan.generate_room_relation()
adjacency = plan.get_adjacency_list()
print(json.dumps(adjacency, indent=2))

graph_svg_path = WORKDIR / 'cubigraph_relations.svg'
graph_svg_path.write_text(str(plan.generate_relation_svg()), encoding='utf-8')
graph_json_path = WORKDIR / 'cubigraph_adjacency.json'
graph_json_path.write_text(json.dumps(adjacency, indent=2), encoding='utf-8')
display(SVG(filename=str(graph_svg_path)))
print('Saved:', graph_svg_path)
print('Saved:', graph_json_path)

## 6. Download outputs

In [ ]:
archive = shutil.make_archive('/content/floorplan_parser_output', 'zip', WORKDIR)
files.download(archive)